In [1]:
import scala.math
import org.apache.spark._
import org.apache.spark.sql._
import org.apache.spark.rdd._
import org.apache.spark.ml.linalg._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.functions.{col}
import org.apache.spark.sql.{DataFrame, Row, SparkSession}
import org.apache.spark.ml.{Pipeline, PipelineModel, PipelineStage}
import org.apache.spark.ml.tuning.{ ParamGridBuilder, CrossValidator}
import org.apache.spark.ml.classification.{MultilayerPerceptronClassifier, RandomForestClassifier}
import org.apache.spark.ml.feature.{StandardScaler, VectorAssembler, StringIndexer, MinMaxScaler}
import org.apache.spark.ml.evaluation.{MulticlassClassificationEvaluator,BinaryClassificationEvaluator}
import org.apache.spark.ml.clustering._

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
574,application_1580996944851_0530,spark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

import scala.math
import org.apache.spark._
import org.apache.spark.sql._
import org.apache.spark.rdd._
import org.apache.spark.ml.linalg._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.{DataFrame, Row, SparkSession}
import org.apache.spark.ml.{Pipeline, PipelineModel, PipelineStage}
import org.apache.spark.ml.tuning.{ParamGridBuilder, CrossValidator}
import org.apache.spark.ml.classification.{MultilayerPerceptronClassifier, RandomForestClassifier}
import org.apache.spark.ml.feature.{StandardScaler, VectorAssembler, StringIndexer, MinMaxScaler}
import org.apache.spark.ml.evaluation.{MulticlassClassificationEvaluator, BinaryClassificationEvaluator}
import org.apache.spark.ml.clustering._


In [2]:
object osbnrClass{

    def Osbnr(data: DataFrame, k: Int) : DataFrame = {

        val Majority = data.filter($"Class" === "0")
        val Minority = data.filter($"Class" === "1")
        val modelKmeans = new KMeans().setK(k).setFeaturesCol("features").setPredictionCol("prediction").setSeed(1L).setMaxIter(100)
        val modelk = modelKmeans.fit(Minority)
        var predictionsMin = modelk.transform(Minority)
        val centroids=modelk.clusterCenters
        var predictionsMaj = modelk.transform(Majority)
        val max = maxByCentroid(centroids, predictionsMin, k)
        val noisydata = getNoisyInstances(centroids, max, predictionsMaj).drop("dist","prediction","noisy").checkpoint()
        val newMajority = Majority.exceptAll(noisydata)
        val newData = newMajority.union(Minority).sort("Time")
        
    return newData
}
    
def distance(centroid: Vector, data: Vector): Double = math.sqrt(centroid.toArray.zip(data.toArray).map(p => p._1 - p._2).map(d => d * d).sum)
def distanceAllCluster(centroid: Vector, dataCentroid: Array[DenseVector]): Array[Double] = {
      dataCentroid.map(d => distance(centroid, d))
    }
//DenseVector
def maxByCentroid(centroids: Array[Vector], data: DataFrame, k: Int): Map[Int, Double] = {
      val max = (0 until k).map{ k => val dataCentroid = data.filter($"prediction" === k).select("features").collect().map {
            case Row(v: Vector) => v.toDense
          }
        val dist = distanceAllCluster(centroids(k), dataCentroid)
        if (dist.isEmpty) {
          (k, 0.0)
        }
        else
          (k, dist.max)
      }
      max.toMap
    }
def calculateDistance(centroids: Array[Vector]) = udf((v: Vector, k: Int) => {
      math.sqrt(centroids(k).toArray.zip(v.toArray)
        .map(p => p._1 - p._2).map(d => d * d).sum)
    })
    
def checkNoisy(max: Map[Int, Double]) = udf((distance: Double, k: Int) => if (distance <= max(k)) 1 else 0)
    
def getNoisyInstances(centroids: Array[Vector], max: Map[Int, Double], PredictionMaj: DataFrame) = {
    val distanceDF = PredictionMaj.withColumn("dist", calculateDistance(centroids)(PredictionMaj("features"), PredictionMaj("prediction"))).checkpoint()
      val noiseinstances = distanceDF.withColumn("noisy", checkNoisy(max)(distanceDF("dist"), distanceDF("prediction"))).checkpoint()
        noiseinstances.filter($"noisy" > 0).dropDuplicates
    }
}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

defined object osbnrClass


In [3]:
sc.setCheckpointDir("checkpoints/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
val raw = spark.read.format("csv").option("header", "true").option("mode", "DROPMALFORMED").csv("datasets/creditcard.csv")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

raw: org.apache.spark.sql.DataFrame = [Time: string, V1: string ... 29 more fields]


In [5]:
// cast all the column to Double type.
val df = raw.select(((1 to 28).map(i => "V" + i) ++ Array("Time", "Amount", "Class")).map(s => col(s).cast("Double")): _*)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

df: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 29 more fields]


In [6]:
// Preparing Data 
val labelConverter = new StringIndexer().setInputCol("Class").setOutputCol("label")
val assembler = new VectorAssembler().setInputCols((1 to 28).map(i => "V" + i).toArray ++ Array("Amount")).setOutputCol("assembled")
val scaler = new StandardScaler().setInputCol("assembled").setOutputCol("features")
val pipeline = new Pipeline().setStages(Array(assembler, scaler, labelConverter))
val pipelineModel = pipeline.fit(df)
val data = pipelineModel.transform(df)
println("Generate feature from raw data:")
data.select("features","label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

labelConverter: org.apache.spark.ml.feature.StringIndexer = strIdx_20ae724c714a
assembler: org.apache.spark.ml.feature.VectorAssembler = vecAssembler_82ff65a2d07f
scaler: org.apache.spark.ml.feature.StandardScaler = stdScal_e5c38a3a8ae1
pipeline: org.apache.spark.ml.Pipeline = pipeline_bc3b5c799106
pipelineModel: org.apache.spark.ml.PipelineModel = pipeline_bc3b5c799106
data: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 32 more fields]
Generate feature from raw data:
+--------------------+-----+
|            features|label|
+--------------------+-----+
|[-0.6942411021638...|  0.0|
|[0.60849525943109...|  0.0|
|[-0.6934992452238...|  0.0|
|[-0.4933240320774...|  0.0|
|[-0.5913287255806...|  0.0|
|[-0.2174742415711...|  0.0|
|[0.62779408220828...|  0.0|
|[-0.3289277697329...|  0.0|
|[-0.4565722152677...|  0.0|
|[-0.1726974406947...|  0.0|
|[0.73980031932134...|  0.0|
|[0.19654824114227...|  0.0|
|[0.63817910856358...|  0.0|
|[0.54596205586179...|  0.0|
|[-1.4253641430361.

In [7]:
val data0 = data.filter($"Class" === "0").cache()
val data1 = data.filter($"Class" === "1").cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

data0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
data1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [8]:
val splitTime0 = data0.stat.approxQuantile("Time", Array(0.7), 0.001).head
val splitTime1 = data1.stat.approxQuantile("Time", Array(0.8), 0.001).head
val trainingData0 = data0.filter(s"Time<$splitTime0").cache()
val validationData0 = data0.filter(s"Time>=$splitTime0").cache()
val trainingData1 = data1.filter(s"Time<$splitTime1").cache()
val validationData1 = data1.filter(s"Time>=$splitTime1").cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

splitTime0: Double = 132988.0
splitTime1: Double = 137211.0
trainingData0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
trainingData1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [9]:
val trainingData = trainingData0.unionAll(trainingData1)
val validationData = validationData0.unionAll(validationData1)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

trainingData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [10]:
println(" Training set statistics: 1 represents fraud and 0 represents normal")
trainingData.groupBy("Class").count().show()
println(" validation set statistics: 1 represents fraud and 0 represents normal")
validationData.groupBy("Class").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

 Training set statistics: 1 represents fraud and 0 represents normal
+-----+------+
|Class| count|
+-----+------+
|  0.0|199097|
|  1.0|   393|
+-----+------+

 validation set statistics: 1 represents fraud and 0 represents normal
+-----+-----+
|Class|count|
+-----+-----+
|  0.0|85218|
|  1.0|   99|
+-----+-----+



# Under-Sampling OSBNR

In [11]:
val s = System.nanoTime
val newtrainingdata = osbnrClass.Osbnr(trainingData, 6)
val durationprediction = (System.nanoTime - s) / 1e9d
println(s"Resampling process takes $durationprediction secs")
println("New training set statistics: 1 represents Minority class and 0 represents Majority class")
newtrainingdata.groupBy("Class").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 30500629464377183
newtrainingdata: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 32 more fields]
durationprediction: Double = 19.220557761
Resampling process takes 19.220557761 secs
New training set statistics: 1 represents Minority class and 0 represents Majority class
+-----+-----+
|Class|count|
+-----+-----+
|  0.0|  938|
|  1.0|  393|
+-----+-----+



# Learning Model 

In [12]:
// Create a RandomForest model.
val RF = new RandomForestClassifier().setLabelCol("label").setFeaturesCol("features").setNumTrees(20).setImpurity("gini").setMaxBins(28).setMaxDepth(8).setFeatureSubsetStrategy("auto")
val t = System.nanoTime

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RF: org.apache.spark.ml.classification.RandomForestClassifier = rfc_d2ebac072298
t: Long = 30500662351720511


In [13]:
// train the RandomForest model.
val Modelrf = RF.fit(newtrainingdata)
val durationtrain = (System.nanoTime - t) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationtrain secs")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Modelrf: org.apache.spark.ml.classification.RandomForestClassificationModel = RandomForestClassificationModel (uid=rfc_d2ebac072298) with 20 trees
durationtrain: Double = 40.013273987

initial model training finished.
Training process takes 40.013273987 secs


In [14]:
val s = System.nanoTime
val predictionsRF = Modelrf.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsRF.select("prediction", "label", "features").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 30500704581414682
predictionsRF: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]
durationprediction: Double = 0.281352309

initial model training finished.
Training process takes 0.281352309 secs
+----------+-----+--------------------+
|prediction|label|            features|
+----------+-----+--------------------+
|       0.0|  0.0|[0.83090551333317...|
|       0.0|  0.0|[-0.3929372163812...|
|       0.0|  0.0|[1.04065182918104...|
|       0.0|  0.0|[-0.9117871681408...|
|       0.0|  0.0|[0.94807610534099...|
+----------+-----+--------------------+
only showing top 5 rows



In [15]:
println(s"Matrice de confusion :")
predictionsRF.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|81853|
|       1.0|  0.0| 3365|
|       0.0|  1.0|   15|
|       1.0|  1.0|   84|
+----------+-----+-----+



In [16]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsRF)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsRF))
println("Accuracy = " + evaluator1.evaluate(predictionsRF))
println("Precision = " + evaluator2.evaluate(predictionsRF))
println("Recall = " + evaluator3.evaluate(predictionsRF))
println("F1 = " + evaluator4.evaluate(predictionsRF))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_cb7dc7e33c0c
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_f921a04f59d4
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_86abe6ccadd1
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_85da6e6b8dcd
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_182701f87a5e
accuracy: Double = 0.9603830420666456
Area Under ROC Curve = 0.904498942818312
Accuracy = 0.9603830420666456
Precision = 0.9986848733631861
Recall = 0.9603830420666456
F1 = 0.9786889361193887
Test Error = 0.0396169579333544


In [ ]:
val rfModel = {
    val rfGridSearch = for (
    rfNumTrees <- Array(10, 15, 20);
    rfImpurity <- Array("entropy","gini");
    rfMaxBins <- Array(24, 28, 32);
    rfmaxDepth <- Array(4, 6, 8))
    yield {
   println(s"Training random forest numTrees : $rfNumTrees, impurity : $rfImpurity, maxBins: $rfMaxBins, maxDepth : $rfmaxDepth")     
   val rfModel = new RandomForestClassifier().setLabelCol("label").setFeaturesCol("features").setNumTrees(rfNumTrees).setImpurity(rfImpurity).setMaxDepth(rfmaxDepth).setMaxBins(rfMaxBins).setFeatureSubsetStrategy("auto").fit(newtrainingdata)
   val predictionsRF = rfModel.transform(validationData)      
   val rfAUC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC").evaluate(predictionsRF)  
   println("Area Under ROC Curve = " + rfAUC)
       ((rfNumTrees, rfImpurity, rfMaxBins, rfmaxDepth), rfModel, rfAUC)
    }  
    println(rfGridSearch.sortBy(-_._3).take(5).mkString("\n"))
        val BestModel = rfGridSearch.sortBy(-_._3).head._2
    BestModel
}   

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
val predictionsBest = rfModel.transform(validationData)

In [ ]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsBest)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsBest))
println("Accuracy = " + evaluator1.evaluate(predictionsBest))
println("Precision = " + evaluator2.evaluate(predictionsBest))
println("Recall = " + evaluator3.evaluate(predictionsBest))
println("F1 = " + evaluator4.evaluate(predictionsBest))
println("Test Error = " + (1.0 - accuracy))

In [ ]:
println(s"Matrice de confusion :")
   predictionsBest.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

In [ ]:
// create Multilayer Perceptron Model and set its parameters
val t = System.nanoTime
val layers = Array[Int] (29, 60, 30, 2)
val mlp = new MultilayerPerceptronClassifier().setLayers(layers).setLabelCol("label").setFeaturesCol("features").setTol(1E-4).setMaxIter(100) 

In [ ]:
// Train a Multilayer Perceptron model.
val modelMLP = mlp.fit(newtrainingdata)
val durationtrain = (System.nanoTime - t) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationtrain secs")

In [ ]:
val s = System.nanoTime
val predictionsMLP = modelMLP.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsMLP.select("prediction","label").show(5)

In [ ]:
println(s"Classified test set :")
validationData.groupBy("Class").count().show()
println(s"Prediction :")
predictionsMLP.groupBy("prediction").count().show()

In [ ]:
println(s"Matrice de confusion :")
predictionsMLP.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

In [ ]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsMLP)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsMLP))
println("Accuracy = " + evaluator1.evaluate(predictionsMLP))
println("Precision = " + evaluator2.evaluate(predictionsMLP))
println("Recall = " + evaluator3.evaluate(predictionsMLP))
println("F1 = " + evaluator4.evaluate(predictionsMLP))
println("Test Error = " + (1.0 - accuracy))

In [ ]:
val mlpModel = {
    val mlpGridSearch = for (
    mlplayers <- Array( Array[Int] (29, 60, 30, 2), Array[Int] (29, 13, 4, 2), Array[Int] (29, 58, 20, 2));
    mlpMaxIter <- Array(25, 50, 100);
    mlpTol <- Array(1E-4, 1E-6)) 
    yield {
   println(s"Training multilayers perception layers : $mlplayers, Tolerance : $mlpTol, Iterations: $mlpMaxIter")    
   val mlpModel = new MultilayerPerceptronClassifier().setLayers(mlplayers).setLabelCol("label").setFeaturesCol("features").setTol(mlpTol).setBlockSize(128).setSeed(1234L).setMaxIter(mlpMaxIter).fit(newtrainingdata) 
   val predictionsMLP = mlpModel.transform(validationData)      
   val mlpAUC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC").evaluate(predictionsMLP)  
   println("Area Under ROC Curve = " + mlpAUC)
            ((mlplayers, mlpMaxIter, mlpTol), mlpModel, mlpAUC)
    }
    
    println(mlpGridSearch.sortBy(-_._3).take(5).mkString("\n"))
        val BestModel = mlpGridSearch.sortBy(-_._3).head._2
    BestModel
}   

In [ ]:
val predictionsBestMLP = mlpModel.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsBestMLP.select("prediction","label").show(5)

In [ ]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsBestMLP)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsBestMLP))
println("Accuracy = " + evaluator1.evaluate(predictionsBestMLP))
println("Precision = " + evaluator2.evaluate(predictionsBestMLP))
println("Recall = " + evaluator3.evaluate(predictionsBestMLP))
println("F1 = " + evaluator4.evaluate(predictionsBestMLP))
println("Test Error = " + (1.0 - accuracy))

In [ ]:
println(s"Matrice de confusion :")
predictionsBestMLP.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()